# Colab 전차 3D Gaussian Splatting 학습

이 노트북은 **Google Colab**에서 전차별 3D 모델을 생성합니다.

## ⚠️ 사전 설정
- **런타임 → 런타임 유형 변경 → GPU** 선택 (T4 이상 권장)
- Google Drive 공유 문서함 또는 MyDrive에 `전차데이터.zip` 업로드

## 흐름
1. Drive 마운트 & 전차데이터 로드
2. 각도별 scene → 전차별 scene 병합
3. COLMAP + 3D Gaussian Splatting 설치 (vocab tree 포함)
4. 전차별 COLMAP → 3D GS 학습 (**전체 데이터** 사용, vocab_tree_matcher)
5. 결과 Drive 저장

## 1. Drive 마운트 & 전차데이터 압축 해제

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
# 전차데이터.zip 위치 (아래 순서로 탐색)
from pathlib import Path
drive = Path("/content/drive")
tanks_base = Path("/content/data/tanks")
# 이미 압축 해제됐는지 확인 (3. 라벨링 존재)
already_extracted = (
    (tanks_base / "전차데이터" / "3. 라벨링").exists()
    or (tanks_base / "전차데이터" / "전차데이터" / "3. 라벨링").exists()
    or (tanks_base / "3. 라벨링").exists()
    or (tanks_base.exists() and any(tanks_base.rglob("3. 라벨링")))
)

if already_extracted:
    print("✓ 이미 압축 해제됨. /content/data/tanks 사용")
    get_ipython().system('find /content/data/tanks -maxdepth 5 -type d -name "*라벨*" 2>/dev/null')
else:
    candidates = [
        drive / "Shareddrives/최종_데이터/전차데이터.zip",
        drive / "MyDrive/데이터 분석과정/한화_최종프로젝트/최종_데이터/전차데이터.zip",
        drive / "MyDrive/hanhwa_final/전차데이터.zip",
    ]
    ZIP_PATH = None
    for p in candidates:
        if p.exists():
            ZIP_PATH = str(p)
            print(f"발견: {ZIP_PATH}")
            break
    if not ZIP_PATH:
        print("⚠️ 전차데이터.zip을 찾을 수 없습니다. 아래 경로 중 하나에 업로드하세요:")
        for p in candidates:
            print(f"  - {p.parent}")
        get_ipython().system('ls -la /content/drive/MyDrive/ 2>/dev/null || true')
    else:
        get_ipython().system('mkdir -p /content/data/tanks')
        get_ipython().system(f'unzip -q "{ZIP_PATH}" -d "/content/data/tanks"')
        print("압축 해제 완료. 라벨링 폴더:")
        get_ipython().system('find /content/data/tanks -maxdepth 5 -type d -name "*라벨*" 2>/dev/null')

✓ 이미 압축 해제됨. /content/data/tanks 사용
/content/data/tanks/전차데이터/3. 라벨링
/content/data/tanks/전차데이터/3. 라벨링/tiger/라벨링
/content/data/tanks/전차데이터/3. 라벨링/K2/라벨링
/content/data/tanks/전차데이터/3. 라벨링/90식/라벨링
/content/data/tanks/전차데이터/3. 라벨링/M1A2/라벨링
/content/data/tanks/전차데이터/3. 라벨링/K1A1/라벨링
/content/data/tanks/전차데이터/3. 라벨링/T-90a/라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/tiger/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/t-90a/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/k1a1/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/m1a2/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/k2/동영상 라벨링
/content/data/tanks/전차데이터/1. 동영상 관련/동영상 라벨링/90식/동영상 라벨링


## 2. 각도별 scene 생성 (18개)

In [29]:
from pathlib import Path
import shutil

def is_image(p: Path) -> bool:
    return p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}

def prepare_3d_scenes(src_root: Path, out_root: Path) -> None:
    """실제 구조: tank/라벨링/pose/서브폴더(0도,45도 등)/*.jpg"""
    count_scenes = 0
    count_images = 0
    for tank_dir in src_root.iterdir():
        if not tank_dir.is_dir():
            continue
        tank_name = tank_dir.name
        mid_dirs = [d for d in tank_dir.iterdir() if d.is_dir()]
        if len(mid_dirs) == 1 and "라벨" in mid_dirs[0].name:
            label_root = mid_dirs[0]
        else:
            label_root = tank_dir
        for pose_dir in label_root.iterdir():
            if not pose_dir.is_dir():
                continue
            pose_name = pose_dir.name
            scene_name = f"{tank_name}_{pose_name}"
            out_images = out_root / scene_name / "images"
            out_images.mkdir(parents=True, exist_ok=True)
            copied_here = 0
            seen_names = set()
            for p in pose_dir.rglob("*"):
                if is_image(p):
                    rel = p.relative_to(pose_dir)
                    # 서브폴더(0도,45도 등)마다 동일 파일명 → 고유 이름
                    dst_name = f"{rel.parts[0]}_{p.name}" if len(rel.parts) > 1 else p.name
                    if dst_name in seen_names:
                        dst_name = f"{rel.parts[0]}_{p.stem}_{len(seen_names)}{p.suffix}"
                    seen_names.add(dst_name)
                    shutil.copy2(p, out_images / dst_name)
                    copied_here += 1
                    count_images += 1
            if copied_here > 0:
                count_scenes += 1
                print(f"[scene] {scene_name}: {copied_here} images")
    print(f"\n총 scene: {count_scenes}, 이미지: {count_images}")

# 라벨링 루트 탐색 (zip: 전차데이터/3.라벨링, 로컬: 전차데이터/전차데이터/3.라벨링)
def find_labeling_root(base: Path) -> Path | None:
    for cand in [
        base / "전차데이터" / "3. 라벨링",           # zip 압축 해제 직후
        base / "전차데이터" / "전차데이터" / "3. 라벨링",
        base / "3. 라벨링",
    ]:
        if cand.exists():
            return cand
    for d in base.rglob("3. 라벨링"):
        if d.is_dir():
            return d
    return None

LABELING = find_labeling_root(Path("/content/data/tanks"))
if LABELING is None:
    print("❌ '3. 라벨링' 폴더를 찾을 수 없습니다. 위 셀(압축 해제)을 먼저 실행하세요.")
    import subprocess
    subprocess.run("find /content/data/tanks -maxdepth 5 -type d 2>/dev/null | head -30", shell=True)
else:
    print(f"라벨링 경로: {LABELING}")
    SCENES_ROOT = Path("/content/data/3d_scenes")
    SCENES_ROOT.mkdir(parents=True, exist_ok=True)
    prepare_3d_scenes(LABELING, SCENES_ROOT)

라벨링 경로: /content/data/tanks/전차데이터/3. 라벨링
[scene] tiger_1촬영90도: 1105 images
[scene] tiger_1촬영60도: 1106 images
[scene] tiger_1촬영30도: 1098 images
[scene] K2_K2_정면도각도_포신: 5400 images
[scene] K2_K2_90도각도_포신: 5441 images
[scene] K2_K2_45도각도_포신: 5335 images
[scene] 90식_90식_정면각도-포신: 5340 images
[scene] 90식_90식_90도각도_포신: 5416 images
[scene] 90식_90식_45도각도_포신: 5301 images
[scene] M1A2_M1A2_정면각도_포신: 5612 images
[scene] M1A2_M1A2_45도각도_포신: 5605 images
[scene] M1A2_M1A2_90도각도_포신: 5523 images
[scene] K1A1_K1A1_정면각도: 5394 images
[scene] K1A1_K1A1_45도각도: 5664 images
[scene] K1A1_K1A1_90도각도: 5526 images
[scene] T-90a_T-90A_45도각도_포신: 5381 images
[scene] T-90a_T-90A_정면각도_포신: 5369 images
[scene] T-90a_T-90A_90도각도_포신: 5610 images

총 scene: 18, 이미지: 85226


## 3. 전차별 scene 병합 (6개)

In [30]:
def prepare_tank_merged_scenes(scenes_root: Path, out_root: Path) -> None:
    tank_to_scenes = {}
    for scene_dir in scenes_root.iterdir():
        if not scene_dir.is_dir():
            continue
        images_dir = scene_dir / "images"
        if not images_dir.exists():
            continue
        tank_name = scene_dir.name.split("_")[0]
        if tank_name not in tank_to_scenes:
            tank_to_scenes[tank_name] = []
        tank_to_scenes[tank_name].append(scene_dir)

    total_images = 0
    for tank_name, scene_dirs in tank_to_scenes.items():
        out_images = out_root / tank_name / "images"
        out_images.mkdir(parents=True, exist_ok=True)
        copied = 0
        seen = set()
        for scene_dir in scene_dirs:
            for p in (scene_dir / "images").rglob("*"):
                if not is_image(p):
                    continue
                dst_name = f"{p.stem}{p.suffix}"
                if dst_name in seen:
                    dst_name = f"{scene_dir.name}_{p.stem}{p.suffix}"
                seen.add(dst_name)
                shutil.copy2(p, out_images / dst_name)
                copied += 1
                total_images += 1
        print(f"[전차] {tank_name}: {copied} images (from {len(scene_dirs)} scenes)")
    print(f"\n총 6개 전차, {total_images} images")

TANK_MERGED_ROOT = Path("/content/data/3d_scenes_by_tank")
TANK_MERGED_ROOT.mkdir(parents=True, exist_ok=True)
prepare_tank_merged_scenes(SCENES_ROOT, TANK_MERGED_ROOT)

[전차] T-90a: 16360 images (from 4 scenes)
[전차] 90식: 16057 images (from 4 scenes)
[전차] M1A2: 16740 images (from 4 scenes)
[전차] K2: 16176 images (from 4 scenes)
[전차] K1A1: 16584 images (from 4 scenes)
[전차] tiger: 3309 images (from 3 scenes)

총 6개 전차, 85226 images


## 3.5 이미지 서브샘플링 (선택 — 빠른 학습용)

**전체 데이터**로 학습하려면 이 셀을 **실행하지 마세요**.  
빠른 테스트만 할 때만 실행 (전차당 80장으로 축소).

In [31]:
# 전체 데이터 사용 시 이 셀 건너뛰기. 빠른 테스트만 할 때만 실행.
USE_SUBSAMPLING = False  # True로 바꾸면 전차당 80장만 사용
MAX_IMAGES_PER_TANK = 80

def subsample_tank_images(tank_root: Path, max_images: int) -> None:
    img_dir = tank_root / "images"
    if not img_dir.exists():
        return
    images = sorted([p for p in img_dir.iterdir() if is_image(p)])
    n = len(images)
    if n <= max_images:
        print(f"  {tank_root.name}: {n}장 유지")
        return
    step = n / max_images
    indices = [int(i * step) for i in range(max_images)]
    keep = [images[i] for i in indices]
    remove = [p for p in images if p not in keep]
    for p in remove:
        p.unlink()
    print(f"  {tank_root.name}: {n} → {len(keep)}장 (삭제 {len(remove)}장)")

if USE_SUBSAMPLING:
    for tank_dir in TANK_MERGED_ROOT.iterdir():
        if tank_dir.is_dir():
            subsample_tank_images(tank_dir, MAX_IMAGES_PER_TANK)
else:
    print("전체 데이터 사용 (서브샘플링 건너뜀)")

전체 데이터 사용 (서브샘플링 건너뜀)


## 4. COLMAP + 3D Gaussian Splatting 설치

### COLMAP Vocabulary Tree (대량 이미지용)

1만 장 이상 이미지는 `exhaustive_matcher` 대신 `vocab_tree_matcher` 사용.  
1M words 트리: 1만~10만 장 규모에 적합.

In [32]:
# COLMAP vocab tree (1만~10만 장용)
VOCAB_TREE_URL = "https://github.com/colmap/colmap/releases/download/3.11.1/vocab_tree_flickr100K_words1M.bin"
VOCAB_TREE_PATH = "/content/vocab_tree_flickr100K_words1M.bin"
from pathlib import Path
if not Path(VOCAB_TREE_PATH).exists():
    import urllib.request
    urllib.request.urlretrieve(VOCAB_TREE_URL, VOCAB_TREE_PATH)
print("Vocab tree:", VOCAB_TREE_PATH)

Vocab tree: /content/vocab_tree_flickr100K_words1M.bin


In [33]:
!sudo apt-get update -y
!sudo apt-get install -y colmap
!colmap -h | head -3

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease                         
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease               
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease      
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease          
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
colmap is alr

In [34]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
%cd gaussian-splatting
!pip install plyfile tqdm -q

/content
fatal: destination path 'gaussian-splatting' already exists and is not an empty directory.
/content/gaussian-splatting


In [35]:
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install . -q
%cd ../simple-knn
!pip install . -q
%cd /content/gaussian-splatting

/content/gaussian-splatting/submodules/diff-gaussian-rasterization
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
/content/gaussian-splatting/submodules/simple-knn
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: Th

## 5. 전차별 3D 모델 학습 (COLMAP → 3D GS)

### 빠른 검증 모드 (QUICK_TEST)

- **QUICK_TEST=True**: tiger만 80장으로 테스트 (~10~20분). 파이프라인 검증용.
- **QUICK_TEST=False**: 전체 6개 전차, 8만+ 이미지. 성공 확인 후 변경.

In [36]:
import os
import subprocess
from pathlib import Path

# Colab headless 환경: COLMAP Qt/OpenGL 오류 방지
os.environ["QT_QPA_PLATFORM"] = "offscreen"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-root"

# 빠른 검증: True면 tiger만 80장으로 테스트 (약 10~20분). False면 전체 6개 전차.
QUICK_TEST = True  # 성공 확인 후 False로 변경해 전체 학습

DATA_ROOT = Path("/content/data/3d_scenes_by_tank")
GS_ROOT = Path("/content/gaussian-splatting")

def run_colmap_and_train(scene_name: str) -> bool:
    scene_root = DATA_ROOT / scene_name
    img_dir = scene_root / "images"
    if not img_dir.exists():
        print(f"[SKIP] {scene_name}: images 폴더 없음")
        return False

    n_imgs = len([p for p in img_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

    # 이전 실행 잔여물 제거
    db_path = scene_root / "database.db"
    if db_path.exists():
        db_path.unlink()
    sparse_root = scene_root / "sparse"
    if sparse_root.exists():
        import shutil
        shutil.rmtree(sparse_root)
    os.makedirs(str(sparse_root), exist_ok=True)

    sparse_dir = scene_root / "sparse" / "0"  # mapper가 생성

    sparse_str = str(sparse_root.resolve())
    sparse_dir_str = str(sparse_dir)
    assert sparse_root.is_dir(), f"sparse 디렉터리 생성 실패: {sparse_root}"

    # COLMAP (대량 이미지: 해상도·특징 수 제한으로 메모리 절약)
    # Colab headless: OpenGL 없음 → SiftExtraction/SiftMatching CPU 모드 강제
    env = dict(os.environ)
    print(f"\n=== [{scene_name}] COLMAP ({n_imgs}장) ===")
    result = subprocess.run([
        "colmap", "feature_extractor",
        "--database_path", str(db_path),
        "--image_path", str(img_dir),
        "--SiftExtraction.use_gpu", "0",
    ], capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print("COLMAP stderr:", (result.stderr or "")[-2000:])
        raise subprocess.CalledProcessError(result.returncode, result.args)
    # 대량 이미지: vocab_tree_matcher (exhaustive는 1만 장 이상 시 메모리 부족)
    vocab_path = "/content/vocab_tree_flickr100K_words1M.bin"
    for cmd_name, args in [
        ("vocab_tree_matcher", ["colmap", "vocab_tree_matcher", "--database_path", str(db_path),
         "--VocabTreeMatching.vocab_tree_path", vocab_path, "--SiftMatching.use_gpu", "0"]),
        ("mapper", ["colmap", "mapper", "--database_path", str(scene_root / "database.db"),
         "--image_path", str(scene_root / "images"), "--output_path", sparse_str]),
        ("model_converter", ["colmap", "model_converter", "--input_path", sparse_dir_str,
         "--output_path", sparse_dir_str, "--output_type", "TXT"]),
    ]:
        r = subprocess.run(args, capture_output=True, text=True, env=env)
        if r.returncode != 0:
            print(f"COLMAP {cmd_name} stderr:", (r.stderr or "")[-2000:])
            raise subprocess.CalledProcessError(r.returncode, args)

    # sparse/0 결과 검증 (cameras.txt, images.txt 필수)
    if not (sparse_dir / "cameras.txt").exists() or not (sparse_dir / "images.txt").exists():
        raise RuntimeError(f"COLMAP mapper 출력 없음: {sparse_dir} (cameras.txt/images.txt 확인)")

    # 3D Gaussian Splatting
    print(f"=== [{scene_name}] 3D Gaussian Splatting ===")
    train_cmd = ["python", "train.py", "-s", str(scene_root), "-m", f"output/{scene_name}"]
    r = subprocess.run(train_cmd, cwd=str(GS_ROOT), env=env, capture_output=True, text=True)
    if r.returncode != 0:
        print("train.py stderr:", (r.stderr or "")[-3000:])
        raise subprocess.CalledProcessError(r.returncode, train_cmd)

    return True

# 학습할 전차 목록
tanks = [d.name for d in DATA_ROOT.iterdir() if d.is_dir() and (d / "images").exists()]
if QUICK_TEST:
    src_tank = "tiger" if "tiger" in tanks else tanks[0]
    # 원본 유지: tiger_quicktest 폴더에 80장만 복사
    import shutil
    quick_dir = DATA_ROOT / "tiger_quicktest"
    quick_dir.mkdir(exist_ok=True)
    (quick_dir / "images").mkdir(exist_ok=True)
    imgs = sorted([p for p in (DATA_ROOT / src_tank / "images").iterdir()
                   if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
    step = max(1, len(imgs) / 80)
    for i in range(80):
        src = imgs[min(int(i * step), len(imgs) - 1)]
        shutil.copy2(src, quick_dir / "images" / src.name)
    tanks = ["tiger_quicktest"]
    print(f"[QUICK_TEST] {src_tank} → tiger_quicktest: 80장 (원본 유지)")
print(f"학습 대상: {tanks}" + (" (빠른 검증 모드)" if QUICK_TEST else ""))

[QUICK_TEST] tiger → tiger_quicktest: 80장 (원본 유지)
학습 대상: ['tiger_quicktest'] (빠른 검증 모드)


In [37]:
# 전차별 순차 학습 (각 전차당 약 6~50분 소요)
for tank in tanks:
    try:
        run_colmap_and_train(tank)
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] {tank}: {e}")

print("\n전차별 3D 모델 학습 완료")


=== [tiger_quicktest] COLMAP (80장) ===
COLMAP mapper stderr: ERROR: Failed to parse options - unrecognised option '--export_path'.

[ERROR] tiger_quicktest: Command '['colmap', 'mapper', '--database_path', '/content/data/3d_scenes_by_tank/tiger_quicktest/database.db', '--image_path', '/content/data/3d_scenes_by_tank/tiger_quicktest/images', '--export_path', '/content/data/3d_scenes_by_tank/tiger_quicktest/sparse']' returned non-zero exit status 1.

전차별 3D 모델 학습 완료


## 6. 결과 확인

In [38]:
!find /content/gaussian-splatting/output -name "point_cloud.ply" 2>/dev/null

## 7. Drive로 결과 저장

In [39]:
import shutil

OUT_DIR = Path("/content/gaussian-splatting/output")
DRIVE_SAVE = Path("/content/drive/MyDrive/hanhwa_final/3d_models_by_tank")

if DRIVE_SAVE.parent.exists():
    if DRIVE_SAVE.exists():
        shutil.rmtree(DRIVE_SAVE)
    shutil.copytree(OUT_DIR, DRIVE_SAVE)
    print("Drive 저장 완료:", DRIVE_SAVE)
else:
    print("Drive 마운트 경로를 확인하세요. 결과는 /content/gaussian-splatting/output 에 있습니다.")

Drive 마운트 경로를 확인하세요. 결과는 /content/gaussian-splatting/output 에 있습니다.
